# Entrega 1 – Coleta e Pré-processamento

Tema 14 – Fake News e Informação Online

Este notebook realiza a primeira etapa do projeto de Ciência de Dados II, contemplando:

- Carregamento do dataset;
- Análise inicial da base;
- Verificação de valores ausentes;
- Verificação de representatividade;
- Limpeza básica dos dados;
- Tratamento de atributos textuais e temporais;
- Transformações simples;
- Comparação entre StandardScaler e MinMaxScaler.

O dataset utilizado é o FactCenter, apresentado no artigo:

**A Comprehensive Dataset of Brazilian Fact-Checking Stories**  
Fonte: https://journals-sol.sbc.org.br/index.php/jidm/article/view/2354

## 1. Importação das bibliotecas

In [1]:
import ast
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler

## 2. Carregamento do dataset e transformações iniciais

Nesta etapa, o dataset será carregado e algumas transformações básicas serão aplicadas:

- Conversão das colunas `publication_date` e `obtained_at` para data;
- Conversão das colunas `authors`, `categories`, `tags` e `rating` de string para lista de strings;
- Criação de colunas auxiliares para facilitar análises posteriores.

In [2]:
DATASET_PATH = "../../dataset/central_de_fatos.csv"

df = pd.read_csv(
    DATASET_PATH,
    sep = ';'
)

df.head()

,url,source_name,title,subtitle,publication_date,text_news,image_link,video_link,authors,categories,tags,obtained_at,rating
0,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Publicações enganam ao associar Bolsonaro à ap...,NaN,2020-07-31,Ancine aprovou em 2019 a captação de 530 mil r...,NaN,NaN,['Projeto Comprova'],['Políticas públicas'],[],2021-07-06,['Enganoso']
1,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Post mostra imagens de outras estradas para af...,NaN,2020-07-31,Parte dos trechos das gravações utilizadas no ...,https://i2.wp.com/projetocomprova.com.br/wp-co...,NaN,['Projeto Comprova'],['Políticas públicas'],[],2021-07-06,['Falso']
2,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Eduardo Bolsonaro posta vídeo antigo sobre lib...,NaN,2020-07-31,"Em uma publicação no Twitter, o deputado usa c...",NaN,NaN,['Projeto Comprova'],['Pandemia'],[],2021-07-06,['Enganoso']
3,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Médica cita estudos não conclusivos para suger...,NaN,2020-07-30,"Procurada pelo Comprova, médica enviou 34 estu...",NaN,NaN,['Projeto Comprova'],['Pandemia'],[],2021-07-06,['Enganoso']
4,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Médica usa informações falsas em vídeo para fa...,NaN,2020-07-29,Ao contrário do que afirma uma médica em vídeo...,NaN,NaN,['Projeto Comprova'],['Pandemia'],[],2021-07-06,['Falso']


### 2.1 Conversão de datas

In [3]:
df['publication_date'] = pd.to_datetime(df['publication_date'], errors = "coerce")
df['obtained_at'] = pd.to_datetime(df['obtained_at'], errors = "coerce")

### 2.2 Conversão de colunas com listas serializadas

As colunas `authors`, `categories`, `tags` e `rating` possuem valores armazenados como texto no formato de lista, por exemplo: `"['Projeto Comprova']"`

Esses campos serão convertidos para listas reais de strings, pois isso facilita análises como contagem, explosão de valores e regras de associação.

In [4]:
def parse_list_column(value):
    """
    Converte strings no formato "['item 1', 'item 2']" para listas de strings.
    Caso o valor esteja vazio, nulo ou inválido, retorna lista vazia.
    """
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return value
    
    if not isinstance(value, str):
        return []
    
    value = value.strip()
    
    if value == "" or value.lower() in ["nan", "none", "null"]:
        return []

    try:
        parsed_value = ast.literal_eval(value)

        if isinstance(parsed_value, list):
            return list({ str(item).strip().lower() for item in parsed_value if str(item).strip() })
        
        return [str(parsed_value).strip()]
    
    except (ValueError, SyntaxError):
        return [value]

In [5]:
df["authors"] = df["authors"].apply(parse_list_column)
df["categories"] = df["categories"].apply(parse_list_column)
df["tags"] = df["tags"].apply(parse_list_column)
df["rating"] = df["rating"].apply(parse_list_column)

## 3. Informações gerais da base

In [6]:
df.head()

,url,source_name,title,subtitle,publication_date,text_news,image_link,video_link,authors,categories,tags,obtained_at,rating
0,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Publicações enganam ao associar Bolsonaro à ap...,NaN,2020-07-31,Ancine aprovou em 2019 a captação de 530 mil r...,NaN,NaN,[projeto comprova],[políticas públicas],[],2021-07-06,[enganoso]
1,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Post mostra imagens de outras estradas para af...,NaN,2020-07-31,Parte dos trechos das gravações utilizadas no ...,https://i2.wp.com/projetocomprova.com.br/wp-co...,NaN,[projeto comprova],[políticas públicas],[],2021-07-06,[falso]
2,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Eduardo Bolsonaro posta vídeo antigo sobre lib...,NaN,2020-07-31,"Em uma publicação no Twitter, o deputado usa c...",NaN,NaN,[projeto comprova],[pandemia],[],2021-07-06,[enganoso]
3,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Médica cita estudos não conclusivos para suger...,NaN,2020-07-30,"Procurada pelo Comprova, médica enviou 34 estu...",NaN,NaN,[projeto comprova],[pandemia],[],2021-07-06,[enganoso]
4,https://projetocomprova.com.br/publica%C3%A7%C...,COMPROVA,Médica usa informações falsas em vídeo para fa...,NaN,2020-07-29,Ao contrário do que afirma uma médica em vídeo...,NaN,NaN,[projeto comprova],[pandemia],[],2021-07-06,[falso]


In [7]:
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.info()

Quantidade de linhas: 11647
Quantidade de colunas: 13
<class 'pandas.DataFrame'>
RangeIndex: 11647 entries, 0 to 11646
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   url               11647 non-null  str           
 1   source_name       11647 non-null  str           
 2   title             11647 non-null  str           
 3   subtitle          6793 non-null   str           
 4   publication_date  11647 non-null  datetime64[us]
 5   text_news         11643 non-null  str           
 6   image_link        7317 non-null   str           
 7   video_link        1752 non-null   str           
 8   authors           11647 non-null  object        
 9   categories        11647 non-null  object        
 10  tags              11647 non-null  object        
 11  obtained_at       11647 non-null  datetime64[us]
 12  rating            11647 non-null  object        
dtypes: datetime64[us](2), object(4), 

In [8]:
df.columns

Index(['url', 'source_name', 'title', 'subtitle', 'publication_date',
       'text_news', 'image_link', 'video_link', 'authors', 'categories',
       'tags', 'obtained_at', 'rating'],
      dtype='str')

## 4. Verificação de valores faltantes

In [9]:
def is_missing_value(value):
    """
    Considera como valor faltante:
    - NaN / None;
    - strings vazias;
    - listas vazias.
    """
    if value is None:
        return True
    
    if isinstance(value, float) and pd.isna(value):
        return True
    
    if isinstance(value, str) and value.strip() == "":
        return True
    
    if isinstance(value, list) and len(value) == 0:
        return True
    
    return False

In [10]:
missing = df.apply(lambda col: col.map(is_missing_value).sum()).sort_values(ascending=False)
missing_percent = (
    df.apply(lambda col: col.map(is_missing_value).mean()) * 100
).sort_values(
    ascending = False
)

missing_df = pd.DataFrame({
    "valores_faltantes": missing,
    "percentual": missing_percent.round(2)
})

missing_df[missing_df["valores_faltantes"] > 0]

,valores_faltantes,percentual
video_link,9895,84.96
tags,6487,55.70
subtitle,4854,41.68
image_link,4330,37.18
categories,3856,33.11
authors,104,0.89
text_news,4,0.03


## 5. Verificação de registros duplicados

In [11]:
duplicados = df.duplicated(subset = ['url']).sum()
print(f"Quantidade de registros duplicados: {duplicados}")

Quantidade de registros duplicados: 0


## 6. Análise de representatividade da base

Nesta etapa, verificamos a distribuição dos registros por agência e por veredito, quando essas colunas estão disponíveis.

In [12]:
source_col = ["source_name"]
verdict_col = ["rating"]

print("Coluna de agência encontrada:", source_col)
print("Coluna de veredito encontrada:", verdict_col)

Coluna de agência encontrada: ['source_name']
Coluna de veredito encontrada: ['rating']


In [13]:
df[source_col].value_counts(dropna = False)

source_name     
boatos              5523
lupa                2574
aos fatos           1679
fato-ou-fake         917
ESTADAO_VERIFICA     593
COMPROVA             361
Name: count, dtype: int64

In [14]:
df.explode(verdict_col)[verdict_col] \
    .value_counts(dropna=False) \
    .head(20)

rating                 
boato                      5523
falso                      4092
fake                        898
verdadeiro                  828
exagerado                   636
verdadeiro, mas             446
enganoso                    343
impreciso                   199
insustentável               183
contraditório               174
fora de contexto            134
de olho                     120
distorcido                  116
fato                        102
nao e bem assim              92
ainda é cedo para dizer      92
subestimado                  84
comprovado                    9
evidência comprovada          6
contexto errado               4
Name: count, dtype: int64

## 7. Limpeza básica dos dados

A limpeza será simples, apenas para atender à primeira entrega:

- Remoção de duplicatas;
- Remoção de registros sem título e sem texto;
- Conversão de campos textuais para string;
- Remoção de espaços extras.

In [15]:
df_clean = df.copy()

df_clean = df_clean.drop_duplicates(subset = ['url'])

text_cols = ["source_name", "title", "subtitle", "text_news", "image_link", "video_link"]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

df_clean.shape

(11647, 13)

In [16]:
mask_title = df_clean["title"].map(lambda *args, **kwargs: not is_missing_value(*args, **kwargs))
mask_text = df_clean["text_news"].map(lambda *args, **kwargs: not is_missing_value(*args, **kwargs))
mask_authors = df_clean["authors"].map(lambda *args, **kwargs: not is_missing_value(*args, **kwargs))

df_clean = df_clean[
    mask_title &
    mask_text &
    mask_authors
]

print(f"Quantidade de linhas após limpeza: {df_clean.shape[0]}")

Quantidade de linhas após limpeza: 11539


## 8. Criação de atributos derivados

Serão criados atributos simples a partir dos textos e metadados:

- Tamanho do título;
- Tamanho do texto;
- Quantidade de palavras no título;
- Quantidade de palavras no texto;
- Indicador de presença de imagem;
- Indicador de presença de vídeo.

In [17]:
df_clean["title_char_count"] = df_clean["title"].str.len()
df_clean["title_word_count"] = df_clean["title"].str.split().str.len()

df_clean["text_char_count"] = df_clean["text_news"].str.len()
df_clean["text_word_count"] = df_clean["text_news"].str.split().str.len()

df_clean["has_image"] = df_clean["image_link"].notna().astype(int)
df_clean["has_video"] = df_clean["video_link"].notna().astype(int)

df_clean.head()

,url,source_name,title,subtitle,publication_date,text_news,image_link,video_link,authors,categories,tags,obtained_at,rating,title_char_count,title_word_count,text_char_count,text_word_count,has_image,has_video
0,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,publicações enganam ao associar bolsonaro à ap...,NaN,2020-07-31,ancine aprovou em 2019 a captação de 530 mil r...,NaN,NaN,[projeto comprova],[políticas públicas],[],2021-07-06,[enganoso],91,14,12096,1990,0,0
1,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,post mostra imagens de outras estradas para af...,NaN,2020-07-31,parte dos trechos das gravações utilizadas no ...,https://i2.wp.com/projetocomprova.com.br/wp-co...,NaN,[projeto comprova],[políticas públicas],[],2021-07-06,[falso],91,14,7981,1293,1,0
2,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,eduardo bolsonaro posta vídeo antigo sobre lib...,NaN,2020-07-31,"em uma publicação no twitter, o deputado usa c...",NaN,NaN,[projeto comprova],[pandemia],[],2021-07-06,[enganoso],66,9,10524,1720,0,0
3,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,médica cita estudos não conclusivos para suger...,NaN,2020-07-30,"procurada pelo comprova, médica enviou 34 estu...",NaN,NaN,[projeto comprova],[pandemia],[],2021-07-06,[enganoso],78,10,12443,1924,0,0
4,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,médica usa informações falsas em vídeo para fa...,NaN,2020-07-29,ao contrário do que afirma uma médica em vídeo...,NaN,NaN,[projeto comprova],[pandemia],[],2021-07-06,[falso],69,12,15013,2396,0,0


## 9. Seleção de atributos numéricos para pré-processamento

Nesta etapa, serão selecionados apenas atributos numéricos simples para testar normalização e padronização.

In [18]:
numeric_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols

['title_char_count',
 'title_word_count',
 'text_char_count',
 'text_word_count',
 'has_image',
 'has_video']

In [19]:
df_numeric = df_clean[numeric_cols].copy()

df_numeric = df_numeric.replace([np.inf, -np.inf], np.nan)
df_numeric = df_numeric.fillna(df_numeric.median(numeric_only = True))

df_numeric.head(20)

,title_char_count,title_word_count,text_char_count,text_word_count,has_image,has_video
0,91,14,12096,1990,0,0
1,91,14,7981,1293,1,0
2,66,9,10524,1720,0,0
3,78,10,12443,1924,0,0
4,69,12,15013,2396,0,0
5,101,19,5967,1014,0,0
6,98,16,13633,2110,1,0
7,95,16,9296,1566,0,0
8,67,11,14457,2314,0,0
9,64,11,11318,1853,0,0


## 10. Verificação simples de outliers

Será usada a regra do intervalo interquartil (IQR) para identificar possíveis outliers nos atributos numéricos.

In [20]:
outlier_summary = {}

for col in df_numeric.columns:
    q1 = df_numeric[col].quantile(0.25)
    q3 = df_numeric[col].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    
    outliers = df_numeric[
        (df_numeric[col] < lower_limit) | (df_numeric[col] > upper_limit)
    ]
    
    outlier_summary[col] = len(outliers)

pd.DataFrame.from_dict(
    outlier_summary, 
    orient = "index", 
    columns = ["qtd_outliers"]
).sort_values(
    by = "qtd_outliers", 
    ascending = False
)

,qtd_outliers
has_video,1751
text_word_count,936
text_char_count,935
title_char_count,187
title_word_count,76
has_image,0


## 11. Tratamento simples de outliers

Para manter a solução simples, os outliers não serão removidos neste momento.

Justificativa: em dados textuais e de checagem de fatos, textos muito longos, curtos ou com muitos metadados podem representar comportamentos relevantes. A remoção será avaliada nas próximas etapas, principalmente durante a análise exploratória e a clusterização.

## 12. Padronização com StandardScaler

O StandardScaler transforma os dados para média próxima de 0 e desvio padrão próximo de 1.

Essa técnica é útil quando os atributos possuem escalas diferentes e será relevante para algoritmos baseados em distância.

In [21]:
standard_scaler = StandardScaler()

df_standard_scaled = pd.DataFrame(
    standard_scaler.fit_transform(df_numeric),
    columns=df_numeric.columns
)

df_standard_scaled.head(20)

,title_char_count,title_word_count,text_char_count,text_word_count,has_image,has_video
0,0.786002,0.320015,2.275688,2.290641,-1.297253,-0.422957
1,0.786002,0.320015,1.090389,1.062675,0.770860,-0.422957
2,-0.807999,-1.485860,1.822884,1.814959,-1.297253,-0.422957
3,-0.042879,-1.124685,2.375640,2.174363,-1.297253,-0.422957
4,-0.616719,-0.402335,3.115912,3.005928,-1.297253,-0.422957
5,1.423602,2.125890,0.510269,0.571136,-1.297253,-0.422957
6,1.232322,1.042365,2.718411,2.502056,0.770860,-0.422957
7,1.041042,1.042365,1.469166,1.543643,-1.297253,-0.422957
8,-0.744239,-0.763510,2.955759,2.861461,-1.297253,-0.422957
9,-0.935519,-0.763510,2.051591,2.049276,-1.297253,-0.422957


## 13. Normalização com MinMaxScaler

O MinMaxScaler transforma os dados para um intervalo fixo, geralmente entre 0 e 1.

Essa técnica ajuda a evitar que atributos com valores muito altos dominem as medidas de distância.

In [22]:
minmax_scaler = MinMaxScaler()

df_minmax_scaled = pd.DataFrame(
    minmax_scaler.fit_transform(df_numeric),
    columns=df_numeric.columns
)

df_minmax_scaled.head()

,title_char_count,title_word_count,text_char_count,text_word_count,has_image,has_video
0,0.348571,0.277778,0.123896,0.123060,0.0,0.0
1,0.348571,0.277778,0.081544,0.079784,1.0,0.0
2,0.205714,0.138889,0.107717,0.106296,0.0,0.0
3,0.274286,0.166667,0.127468,0.118962,0.0,0.0
4,0.222857,0.222222,0.153918,0.148268,0.0,0.0


## 14. Comparação entre os dados originais, padronizados e normalizados

In [23]:
comparacao = pd.DataFrame({
    "original_media": df_numeric.mean(),
    "original_desvio": df_numeric.std(),
    "standard_media": df_standard_scaled.mean(),
    "standard_desvio": df_standard_scaled.std(),
    "minmax_min": df_minmax_scaled.min(),
    "minmax_max": df_minmax_scaled.max()
})

comparacao

,original_media,original_desvio,standard_media,standard_desvio,minmax_min,minmax_max
title_char_count,78.672502,15.684485,-6.404060e-17,1.000043,0.0,1.0
title_word_count,13.113961,2.768861,1.674908e-16,1.000043,0.0,1.0
text_char_count,4195.499610,3471.847229,6.896680e-17,1.000043,0.0,1.0
text_word_count,689.820348,567.629636,1.034502e-16,1.000043,0.0,1.0
has_image,0.627264,0.483554,-7.881920e-17,1.000043,0.0,1.0
has_video,0.151746,0.358790,5.911440e-17,1.000043,0.0,1.0


## 15. Base pré-processada

Nesta etapa, será criada uma versão simples contendo:

- Dados limpos;
- Atributos derivados;
- Atributos numéricos padronizados;
- Atributos numéricos normalizados.

In [24]:
df_preprocessado = df_clean.copy()

for col in df_standard_scaled.columns:
    df_preprocessado[f"{col}_standard"] = df_standard_scaled[col].values

for col in df_minmax_scaled.columns:
    df_preprocessado[f"{col}_minmax"] = df_minmax_scaled[col].values

df_preprocessado.head(20)

,url,source_name,title,subtitle,publication_date,text_news,image_link,video_link,authors,categories,...,text_char_count_standard,text_word_count_standard,has_image_standard,has_video_standard,title_char_count_minmax,title_word_count_minmax,text_char_count_minmax,text_word_count_minmax,has_image_minmax,has_video_minmax
0,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,publicações enganam ao associar bolsonaro à ap...,NaN,2020-07-31,ancine aprovou em 2019 a captação de 530 mil r...,NaN,NaN,[projeto comprova],[políticas públicas],...,2.275688,2.290641,-1.297253,-0.422957,0.348571,0.277778,0.123896,0.123060,0.0,0.0
1,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,post mostra imagens de outras estradas para af...,NaN,2020-07-31,parte dos trechos das gravações utilizadas no ...,https://i2.wp.com/projetocomprova.com.br/wp-co...,NaN,[projeto comprova],[políticas públicas],...,1.090389,1.062675,0.770860,-0.422957,0.348571,0.277778,0.081544,0.079784,1.0,0.0
2,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,eduardo bolsonaro posta vídeo antigo sobre lib...,NaN,2020-07-31,"em uma publicação no twitter, o deputado usa c...",NaN,NaN,[projeto comprova],[pandemia],...,1.822884,1.814959,-1.297253,-0.422957,0.205714,0.138889,0.107717,0.106296,0.0,0.0
3,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,médica cita estudos não conclusivos para suger...,NaN,2020-07-30,"procurada pelo comprova, médica enviou 34 estu...",NaN,NaN,[projeto comprova],[pandemia],...,2.375640,2.174363,-1.297253,-0.422957,0.274286,0.166667,0.127468,0.118962,0.0,0.0
4,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,médica usa informações falsas em vídeo para fa...,NaN,2020-07-29,ao contrário do que afirma uma médica em vídeo...,NaN,NaN,[projeto comprova],[pandemia],...,3.115912,3.005928,-1.297253,-0.422957,0.222857,0.222222,0.153918,0.148268,0.0,0.0
5,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,"imagem de doria tomando vacina é de março, ant...",NaN,2020-07-27,imagem que circula nas redes sociais é de um t...,NaN,NaN,[projeto comprova],[pandemia],...,0.510269,0.571136,-1.297253,-0.422957,0.405714,0.416667,0.060816,0.062461,0.0,0.0
6,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,site engana ao afirmar que baixa mortalidade p...,NaN,2020-07-24,texto distorce declarações de especialista cub...,https://i0.wp.com/projetocomprova.com.br/wp-co...,NaN,[projeto comprova],[pandemia],...,2.718411,2.502056,0.770860,-0.422957,0.388571,0.333333,0.139715,0.130510,1.0,0.0
7,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,vacinas para covid-19 que chegaram ao brasil s...,NaN,2020-07-24,publicação no instagram comemorava a chegada d...,NaN,NaN,[projeto comprova],[pandemia],...,1.469166,1.543643,-1.297253,-0.422957,0.371429,0.333333,0.095078,0.096734,0.0,0.0
8,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,tuíte engana ao afirmar que vacinas usam célul...,NaN,2020-07-24,linhagens celulares desenvolvidas a partir de ...,NaN,NaN,[projeto comprova],[pandemia],...,2.955759,2.861461,-1.297253,-0.422957,0.211429,0.194444,0.148196,0.143176,0.0,0.0
9,https://projetocomprova.com.br/publica%C3%A7%C...,comprova,brasil não terá uma nova moeda lastreada em ni...,NaN,2020-07-24,"o banco central (bc), responsável exclusivo pe...",NaN,NaN,[projeto comprova],[políticas públicas],...,2.051591,2.049276,-1.297253,-0.422957,0.194286,0.194444,0.115889,0.114554,0.0,0.0


## 16. Resumo da Entrega 1

Nesta etapa foram realizadas as seguintes atividades:

- Carregamento do dataset FactCenter;
- Verificação da quantidade de registros e atributos;
- Análise de valores faltantes;
- Verificação de registros duplicados;
- Análise inicial da representatividade por agência e veredito;
- Limpeza básica dos campos textuais;
- Conversão de datas;
- Criação de atributos derivados;
- Identificação simples de outliers;
- Aplicação de StandardScaler;
- Aplicação de MinMaxScaler.